In [1]:
import numpy as np
import pandas as pd
from brainiak.isc import isc
from voxelwise_tutorials.delayer import Delayer
from himalaya.scoring import correlation_score
from nilearn.input_data import NiftiLabelsMasker
from nilearn.datasets import fetch_atlas_schaefer_2018
from himalaya.kernel_ridge import KernelRidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
import os

def check_all_ts_consistency(title, subject_list, array_obj):
    print(f"\n===== {title} =====")
    print("Subjects from list:", subject_list)
    print("Subjects count from list:", len(subject_list))
    print("Array shape (subjects, TR, parcels):", array_obj.shape)
    
    if len(subject_list) == array_obj.shape[0]:
        print("✅ Subject count MATCHES array dimension")
    else:
        print("❌ Subject count DOES NOT match array → mismatch!")
    
    # Subject indexと順序のペアを可視化
    print("\nIndex → Subject mapping:")
    for idx, s in enumerate(subject_list):
        print(f"  [{idx}] {s}")

# ============================================================
# 設定
# ============================================================
task_train = "bronx"   # 学習に使う物語
task_test  = "piemanpni"       # 評価に使う別物語
base_dir = os.getcwd()
label_path = os.path.join(os.path.dirname(base_dir),
                          'atlas_label/Schaefer2018_400Parcels_7Networks_order_info.txt')

# ============================================================
# 7ネットワークラベル読み込み
# ============================================================
def load_network_labels_7networks(label_fn):
    networks = []
    with open(label_fn, 'r') as f:
        for line in f:
            if '7Networks_' in line:
                network_name = line.split('_')[2]
                networks.append(network_name)
    unique_network_labels = sorted(set(networks), key=networks.index)
    network_indices = [
        int(np.median([i for i, n in enumerate(networks) if n == network]))
        for network in unique_network_labels
    ]
    return networks, unique_network_labels, network_indices

networks, unique_networks, indices = load_network_labels_7networks(label_path)

# ============================================================
# Atlas
# ============================================================
n_parcels = 400
atlas = fetch_atlas_schaefer_2018(n_rois=n_parcels, yeo_networks=17, resolution_mm=2)
masker = NiftiLabelsMasker(atlas.maps, labels=atlas.labels, standardize=True)

# ============================================================
# 学習用データ（piemanpni）読み込み
# ============================================================
model_list = ["gpt2", "gpt_oss", "llama", "llama3", "gte", "w2v"]
# 変数名を動的に選択


df_all = pd.read_csv(f"../parcels_csv/{task_train}_clean-parcels_all_subjects.csv")
valid_subs = sorted(df_all["sub"].unique())
n_TR = df_all["TR"].max() + 1
n_parcels = len([c for c in df_all.columns if c.startswith("parcel_")])

all_ts = np.zeros((len(valid_subs), n_TR, n_parcels))
for i, subname in enumerate(valid_subs):
    sub_df = df_all[df_all["sub"] == subname].sort_values("TR")
    all_ts[i] = sub_df[[f"parcel_{j}" for j in range(n_parcels)]].values
   

    
    
df_piemanpni = pd.read_csv(f"../parcels_csv/{task_test}_clean-parcels_all_subjects.csv")
valid_subs_piemanpni = sorted(df_piemanpni["sub"].unique())
n_TR_piemanpni = df_piemanpni["TR"].max() + 1
n_parcels = len([c for c in df_piemanpni.columns if c.startswith("parcel_")])

all_ts_piemanpni = np.zeros((len(valid_subs_piemanpni), n_TR_piemanpni, n_parcels))
for i, subname in enumerate(valid_subs_piemanpni):
    sub_df = df_piemanpni[df_piemanpni["sub"] == subname].sort_values("TR")
    all_ts_piemanpni[i] = sub_df[[f"parcel_{j}" for j in range(n_parcels)]].values



# --- 学習データから sub-280 を削除 ---
if "sub-280" in valid_subs:
    remove_idx = valid_subs.index("sub-280")
    all_ts = np.delete(all_ts, remove_idx, axis=0)
    print(remove_idx)
    print(1)
    # valid_subs.remove("sub-280")

# # --- piemanpni も削除 ---
# if "sub-280" in valid_subs_piemanpni:
#     remove_idx_p = valid_subs_piemanpni.index("sub-280")
#     all_ts_piemanpni = np.delete(all_ts_piemanpni, remove_idx_p, axis=0)
#     print(remove_idx)
#     print(2)
#     # valid_subs_piemanpni.remove("sub-280")
    
for m_name in model_list:
        
    df_all_pred = pd.read_csv(f"../predicted_all_csv/{m_name}_{task_train}_clean_predicted.csv")
    valid_subs = sorted(df_all_pred["subject"].unique())
    n_TR = df_all_pred["TR"].max() + 1
    n_parcels = len([c for c in df_all_pred.columns if c.startswith("parcel_")])
    
    all_ts_pred = np.zeros((len(valid_subs), n_TR, n_parcels))
    for i, subname in enumerate(valid_subs):
        sub_df = df_all_pred[df_all_pred["subject"] == subname].sort_values("TR")
        all_ts_pred[i] = sub_df[[f"parcel_{j}" for j in range(n_parcels)]].values
    


    
    df_piemanpni_pred = pd.read_csv(f"../predicted_all_csv/{m_name}_{task_train}_to_{task_test}_clean_predicted.csv")
    valid_subs_piemanpni = sorted(df_piemanpni_pred["subject"].unique())
    n_TR_piemanpni = df_piemanpni_pred["TR"].max() + 1
    n_parcels = len([c for c in df_piemanpni_pred.columns if c.startswith("parcel_")])
    
    all_ts_piemanpni_pred = np.zeros((len(valid_subs_piemanpni), n_TR_piemanpni, n_parcels))
    for i, subname in enumerate(valid_subs_piemanpni):
        sub_df = df_piemanpni_pred[df_piemanpni_pred["subject"] == subname].sort_values("TR")
        all_ts_piemanpni_pred[i] = sub_df[[f"parcel_{j}" for j in range(n_parcels)]].values

    if "sub-280" in valid_subs:
        idx_rm = valid_subs.index("sub-280")
        all_ts_pred = np.delete(all_ts_pred, idx_rm, axis=0)
        valid_subs.remove("sub-280")
        print(remove_idx)
        print(3)
        
    # if "sub-280" in valid_subs_piemanpni:
    #     remove_idx_p = valid_subs_piemanpni.index("sub-280")
    #     all_ts_piemanpni_pred = np.delete(all_ts_piemanpni_pred, remove_idx_p, axis=0)
    #     valid_subs_piemanpni.remove("sub-280")
    #     print(remove_idx)
    #     print(4)
        


        
    print("✅ all_ts shape =", all_ts.shape)
    print("✅ all_ts_pred shape =", all_ts_pred.shape)
    # subjects, TR, parcel = all_ts.shape
    # all_ts_2d = all_ts.reshape(subjects, TR * parcel)
    
    # scaler = preprocessing.MinMaxScaler()
    # # scaler = MinMaxScaler()
    # all_ts_scaled_2d = scaler.fit_transform(all_ts_2d)
    
    # all_ts_scaled = all_ts_scaled_2d.reshape(subjects, TR, parcel)
    # subjects, TR, parcel = all_ts_piemanpni.shape
    # all_ts_piemanpni_2d = all_ts_piemanpni.reshape(subjects, TR * parcel)
    # # scaler = MinMaxScaler()
    # all_ts_piemanpni_scaled_2d = scaler.fit_transform(all_ts_piemanpni_2d)
    # all_ts_piemanpni_scaled = all_ts_piemanpni_scaled_2d.reshape(subjects, TR, parcel)
    # ============================================================
    # ISC 計算
    # ============================================================
    all_ts_t = np.transpose(all_ts, (1, 2, 0))
    isc_all = isc(all_ts_t, pairwise=False)
    
    # ============================================================
    # モデル設定
    # ============================================================
    outer_cv = KFold(n_splits=2)
    inner_cv = KFold(n_splits=5)
    
    # Mean-center each feature (columns of predictor matrix)
    scaler = StandardScaler(with_mean=True, with_std=True)
    
    # Create delays at 3, 4.5, 6, 7.5 seconds (1.5 s TR)
    delayer = Delayer(delays=[2, 3, 4, 5])
    
    # Ridge regression with alpha grid and nested CV
    alphas = np.logspace(1, 10, 10)
    #alphas = np.logspace(-3,4,20)
    ridge = KernelRidgeCV(alphas=alphas, cv=inner_cv)
    
    # Chain transfroms and estimator into pipeline
    pipeline = make_pipeline(scaler, delayer, ridge)
    
    
    # ============================================================
    # 被験者ループ
    # ============================================================
    
    results = []
    for idx_self, subname in enumerate(valid_subs):
        print(f"=== Subject: {subname} ({idx_self+1}/{len(valid_subs)}) ===")
        print(subname)
        # ====== 学習物語 (piemanpni) ======
        other_idx = [i for i in range(len(valid_subs)) if i != idx_self]
        others_mean = all_ts[other_idx].mean(axis=0)
        Y_parcels = all_ts[idx_self]  # (TR, parcels)
        Y_predicted = all_ts_pred[idx_self]
        # 各被験者の埋め込み（piemanpni）
    
        # ---- Cross-validation training ----
        # Y_predicted = []
        # for train, test in outer_cv.split(Y_parcels):
        #     pipeline.fit(X[train], Y_parcels[train])
        #     pred = pipeline.predict(X[test])
        #     Y_predicted.append(pred)
        # Y_predicted = np.vstack(Y_predicted)
    
        corr_each_other = []
        for s in other_idx:
            Y_other_s = all_ts[s]  # (TR, parcels)
            corr_s = correlation_score(Y_other_s, Y_predicted)  # (parcels,)
            corr_each_other.append(corr_s)
    
        pred_vs_group = np.mean(corr_each_other, axis=0)
        #pred_vs_group = correlation_score(others_mean, Y_predicted)
        subj_vs_group = isc_all[idx_self]
        pred_vs_actual = correlation_score(Y_parcels, Y_predicted)
    
        # ============================================================
        # 別物語 (piemanpni) の評価
        # ============================================================
        
        
        if subname in valid_subs_piemanpni:
            idx_piemanpni = valid_subs_piemanpni.index(subname)
            # others_piemanpni_mean = all_ts_piemanpni[other_idx].mean(axis=0)
            Y_parcels_neo = all_ts_piemanpni[idx_self]
            Y_pred_neo = all_ts_piemanpni_pred[idx_self]
    
    
            corr_each_other_story = []
            
    
            for s in other_idx:
                Y_other_s_neo = all_ts_piemanpni[s]  # (TR, parcels)
                corr_s_neo = correlation_score(Y_other_s_neo, Y_pred_neo)
                corr_each_other_story.append(corr_s_neo)
                
            pred_vs_piemanpni_group = np.mean(corr_each_other_story, axis=0)
            #pred_vs_piemanpni_group = correlation_score(others_piemanpni_mean, Y_pred_neo)
            pred_vs_other_story = correlation_score(Y_parcels_neo, Y_pred_neo)
        else:
            print(f"Warning: {subname} not found in piemanpni dataset, filling with NaN")
            pred_vs_other_story = np.full(n_parcels, np.nan)
    
        # ============================================================
        # DataFrameにまとめる
        # ============================================================
        df_sub = pd.DataFrame({
            "subject": subname,
            "parcel": np.arange(n_parcels),
            "pred_vs_group": pred_vs_group,
            "subj_vs_group": subj_vs_group,
            "pred_vs_actual": pred_vs_actual,
            "pred_vs_other_story": pred_vs_other_story,
            "pred_vs_other_story_group": pred_vs_piemanpni_group,
            "network": networks
        })
        results.append(df_sub)
    
    # ============================================================
    # 出力
    # ============================================================
    df_allsubs = pd.concat(results, ignore_index=True)
    output_path = f"../analysis_corr/{m_name}_{task_train}_allsubjects_with_{task_test}_clean_corr_df.csv"
    df_allsubs.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"✅ 出力完了: {output_path}")
print("\n\n🎉 全モデルの処理が完了しました！")

/tmp/ipykernel_960167/511705294.py:6: DeprecationWarning: The import path 'nilearn.input_data' is deprecated in version 0.9. Importing from 'nilearn.input_data' will be possible at least until release 0.13.0. Please import from 'nilearn.maskers' instead.
  from nilearn.input_data import NiftiLabelsMasker


[fetch_atlas_schaefer_2018] Dataset found in /home/y-sato/nilearn_data/schaefer_2018
11
1
✅ all_ts shape = (46, 357, 400)
✅ all_ts_pred shape = (46, 357, 400)
=== Subject: sub-127 (1/46) ===
sub-127
=== Subject: sub-265 (2/46) ===
sub-265
=== Subject: sub-267 (3/46) ===
sub-267
=== Subject: sub-272 (4/46) ===
sub-272
=== Subject: sub-273 (5/46) ===
sub-273
=== Subject: sub-274 (6/46) ===
sub-274
=== Subject: sub-275 (7/46) ===
sub-275
=== Subject: sub-276 (8/46) ===
sub-276
=== Subject: sub-277 (9/46) ===
sub-277
=== Subject: sub-278 (10/46) ===
sub-278
=== Subject: sub-279 (11/46) ===
sub-279
=== Subject: sub-281 (12/46) ===
sub-281
=== Subject: sub-282 (13/46) ===
sub-282
=== Subject: sub-283 (14/46) ===
sub-283
=== Subject: sub-284 (15/46) ===
sub-284
=== Subject: sub-285 (16/46) ===
sub-285
=== Subject: sub-286 (17/46) ===
sub-286
=== Subject: sub-287 (18/46) ===
sub-287
=== Subject: sub-288 (19/46) ===
sub-288
=== Subject: sub-289 (20/46) ===
sub-289
=== Subject: sub-290 (21/46) =

In [10]:
all_ts

array([[[0.29128217, 0.2335268 , 0.36856827, ..., 0.62448079,
         0.44033538, 0.59431992],
        [0.23478911, 0.23639084, 0.40218075, ..., 0.59559468,
         0.58097736, 0.62174263],
        [0.32689328, 0.38753487, 0.43350477, ..., 0.66835456,
         0.64566935, 0.71521147],
        ...,
        [0.53830582, 0.60306939, 0.66331899, ..., 0.29660854,
         0.66597488, 0.56190553],
        [0.52580468, 0.50806986, 0.69658294, ..., 0.41580569,
         0.63443   , 0.45259692],
        [0.67088607, 0.59117085, 0.6990807 , ..., 0.42789893,
         0.85586412, 0.70020409]],

       [[0.1874405 , 0.27127582, 0.13368724, ..., 0.21261703,
         0.13863787, 0.33447779],
        [0.22162096, 0.30245393, 0.11672968, ..., 0.        ,
         0.36195038, 0.35076962],
        [0.29947582, 0.38276376, 0.13890773, ..., 0.22940194,
         0.5091477 , 0.48527653],
        ...,
        [0.34376492, 0.18499851, 0.4037701 , ..., 0.91103445,
         0.56499503, 0.6639508 ],
        [0.3